# Covid/Ukr Cross-Graph Eval Results

Reads `eval_results.csv` from this directory and plots inline. The main plot is a graph/task grid with shots on the x-axis and a configurable metric on the y-axis.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

CSV_PATH = Path("eval_results.csv")

# Swap these when needed.
SPLIT = "test"
METRIC = "roc_auc"

DATASET_ORDER = [
    "covid19_twitter",
    "ukr_rus_twitter",
    "midterm",
    "covid_political",
    "election2020",
    "ukr_rus_suspended",
]

TASK_ORDER = ["nm", "lp", "pl"]
TASK_LABELS = {
    "nm": "Neighbor matching",
    "lp": "Temporal link prediction",
    "pl": "Classification",
}
METRICS = ["accuracy", "f1", "roc_auc"]

In [ ]:
df = pd.read_csv(CSV_PATH)
df["shots"] = df["shots"].astype(int)

for col in METRICS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df["dataset"] = pd.Categorical(df["dataset"], categories=DATASET_ORDER, ordered=True)
df["task"] = pd.Categorical(df["task"], categories=TASK_ORDER, ordered=True)
df["task_label"] = df["task"].astype(str).map(TASK_LABELS).fillna(df["task"].astype(str))
df = df.sort_values(["split", "dataset", "task", "checkpoint", "shots"])

display(df.head())
display(
    df.groupby(["split", "dataset", "task", "checkpoint"], observed=True)
      .agg(shots=("shots", lambda x: sorted(set(x))), rows=("shots", "size"))
      .reset_index()
)

In [ ]:
def checkpoint_sort_key(value):
    value = str(value)
    suffix_scale = {"": 1, "k": 1_000, "m": 1_000_000}
    suffix = value[-1].lower() if value[-1:].lower() in suffix_scale else ""
    number = value[:-1] if suffix else value
    return (int(number) * suffix_scale[suffix], value) if number.isdigit() else (10**18, value)


def plot_metric_grid(data, *, split=SPLIT, metric=METRIC, datasets=DATASET_ORDER, tasks=TASK_ORDER):
    subset = data[(data["split"] == split) & data[metric].notna()].copy()
    if subset.empty:
        raise ValueError(f"No rows for split={split!r}, metric={metric!r}")

    checkpoints = sorted(subset["checkpoint"].dropna().astype(str).unique(), key=checkpoint_sort_key)
    palette = dict(zip(checkpoints, sns.color_palette("tab10", n_colors=len(checkpoints))))

    fig, axes = plt.subplots(
        nrows=len(datasets),
        ncols=len(tasks),
        figsize=(4.6 * len(tasks), 2.7 * len(datasets)),
        sharex=True,
        sharey=True,
    )

    if len(datasets) == 1 and len(tasks) == 1:
        axes = [[axes]]
    elif len(datasets) == 1:
        axes = [axes]
    elif len(tasks) == 1:
        axes = [[ax] for ax in axes]

    for row_idx, dataset in enumerate(datasets):
        for col_idx, task in enumerate(tasks):
            ax = axes[row_idx][col_idx]
            panel = subset[(subset["dataset"].astype(str) == dataset) & (subset["task"].astype(str) == task)]
            task_label = TASK_LABELS.get(task, task)
            ax.set_title(f"{dataset}\n{task_label}", fontsize=10)

            if panel.empty:
                ax.text(0.5, 0.5, "no data", transform=ax.transAxes, ha="center", va="center", color="0.45")
            else:
                for checkpoint in checkpoints:
                    line = panel[panel["checkpoint"].astype(str) == checkpoint].sort_values("shots")
                    if line.empty:
                        continue
                    ax.plot(
                        line["shots"],
                        line[metric],
                        marker="o",
                        linewidth=2,
                        markersize=5,
                        label=checkpoint,
                        color=palette[checkpoint],
                    )

            ax.set_ylim(0, 1.02)
            ax.set_xticks(sorted(subset["shots"].dropna().unique()))
            ax.grid(True, alpha=0.3)
            if row_idx == len(datasets) - 1:
                ax.set_xlabel("shots")
            else:
                ax.set_xlabel("")
            if col_idx == 0:
                ax.set_ylabel(metric)
            else:
                ax.set_ylabel("")

    handles, labels = [], []
    for ax in fig.axes:
        h, l = ax.get_legend_handles_labels()
        for handle, label in zip(h, l):
            if label not in labels:
                handles.append(handle)
                labels.append(label)

    if handles:
        fig.legend(handles, labels, title="checkpoint", loc="upper center", ncol=len(labels), bbox_to_anchor=(0.5, 1.01))

    fig.suptitle(f"{split} {metric} by graph, task, and shots", y=1.03, fontsize=14)
    fig.tight_layout()
    plt.show()


plot_metric_grid(df, split=SPLIT, metric=METRIC)

In [ ]:
# Compact table for the currently selected split/metric.
table = (
    df[df["split"] == SPLIT]
    [["dataset", "task", "checkpoint", "shots", METRIC]]
    .sort_values(["dataset", "task", "checkpoint", "shots"])
)
display(table)